In [8]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Load the filtered leaderboard you already created
# ------------------------------------------------------------
df = pd.read_csv("lr_filtered_accuracy_lt_092_full.csv")

# If needed, round accuracy so "different" means different at 4 decimals
df["Test Accuracy"] = df["Test Accuracy"].round(4)
df["Test ROC-AUC"] = df["Test ROC-AUC"].round(4)
df["Test Precision"] = df["Test Precision"].round(4)
df["Test Recall"] = df["Test Recall"].round(4)
df["Test F1 Score"] = df["Test F1 Score"].round(4)
df["Test Log Loss"] = df["Test Log Loss"].round(4)

# ------------------------------------------------------------
# 2. Keep only 4 core model families
#    (ignore Balanced versions to avoid redundancy)
# ------------------------------------------------------------
def get_core_family(model_name):
    if model_name.startswith("Standard LR"):
        return "Standard LR"
    elif model_name.startswith("Ridge LR") and "Balanced" not in model_name:
        return "Ridge LR"
    elif model_name.startswith("LASSO LR") and "Balanced" not in model_name:
        return "LASSO LR"
    elif model_name.startswith("Elastic-Net LR") and "Balanced" not in model_name:
        return "Elastic-Net LR"
    else:
        return None

df["Core Family"] = df["Model"].apply(get_core_family)
df = df[df["Core Family"].notna()].copy()

# ------------------------------------------------------------
# 3. Sort by performance so best candidates come first
# ------------------------------------------------------------
df = df.sort_values(
    by=["Test ROC-AUC", "Test F1 Score", "Test Accuracy", "Non-zero Coefficients"],
    ascending=[False, False, False, True]
).reset_index(drop=True)

# ------------------------------------------------------------
# 4. Pick one model per family, but all 4 must have different accuracy
# ------------------------------------------------------------
target_families = ["Standard LR", "Ridge LR", "LASSO LR", "Elastic-Net LR"]

selected_rows = []
used_accuracies = set()

for fam in target_families:
    fam_df = df[df["Core Family"] == fam].copy()

    chosen = None
    for _, row in fam_df.iterrows():
        acc = row["Test Accuracy"]
        if acc not in used_accuracies:
            chosen = row
            used_accuracies.add(acc)
            break

    if chosen is not None:
        selected_rows.append(chosen)

selected_df = pd.DataFrame(selected_rows)

# ------------------------------------------------------------
# 5. If one family could not be selected due to duplicate accuracy,
#    choose the next best family candidate with different accuracy
# ------------------------------------------------------------
# This block tries again more flexibly if fewer than 4 models are selected.
if len(selected_df) < 4:
    remaining_families = [f for f in target_families if f not in selected_df["Core Family"].tolist()]

    for fam in remaining_families:
        fam_df = df[df["Core Family"] == fam].copy()

        for _, row in fam_df.iterrows():
            acc = row["Test Accuracy"]
            if acc not in used_accuracies:
                selected_df = pd.concat([selected_df, row.to_frame().T], ignore_index=True)
                used_accuracies.add(acc)
                break

# ------------------------------------------------------------
# 6. Final sorting for presentation
# ------------------------------------------------------------
selected_df = selected_df.sort_values(
    by=["Test ROC-AUC", "Test F1 Score", "Test Accuracy"],
    ascending=[False, False, False]
).reset_index(drop=True)

# ------------------------------------------------------------
# 7. Print final 4-model summary
# ------------------------------------------------------------
print("\n" + "=" * 120)
print("FINAL 4 REPRESENTATIVE LR MODELS (core families only, all with different test accuracy)")
print("=" * 120)

print(selected_df[[
    "Core Family",
    "Model",
    "Non-zero Coefficients",
    "Test ROC-AUC",
    "Test Accuracy",
    "Test Precision",
    "Test Recall",
    "Test F1 Score",
    "Test Log Loss"
]].to_string(index=False))

# ------------------------------------------------------------
# 8. Save final table
# ------------------------------------------------------------
final_table = selected_df[[
    "Core Family",
    "Model",
    "Non-zero Coefficients",
    "Test ROC-AUC",
    "Test Accuracy",
    "Test Precision",
    "Test Recall",
    "Test F1 Score",
    "Test Log Loss"
]].copy()

final_table.to_csv("table3_final_4_lr_models_unique_accuracy.csv", index=False)

with open("table3_final_4_lr_models_unique_accuracy.md", "w", encoding="utf-8") as f:
    f.write(final_table.to_markdown(index=False))

print("\nSaved files:")
print("- table3_final_4_lr_models_unique_accuracy.csv")
print("- table3_final_4_lr_models_unique_accuracy.md")


FINAL 4 REPRESENTATIVE LR MODELS (core families only, all with different test accuracy)
   Core Family                                                   Model  Non-zero Coefficients  Test ROC-AUC  Test Accuracy  Test Precision  Test Recall  Test F1 Score  Test Log Loss
      Ridge LR                             Ridge LR (L2, lbfgs, C=0.1)                     47        0.9722         0.9143          0.9167       0.9167         0.9167         0.2560
   Standard LR                Standard LR (No Penalty, Intercept=True)                     47        0.9485         0.9000          0.9143       0.8889         0.9014         0.2719
Elastic-Net LR Elastic-Net LR (C=0.01, l1_ratio=0.25, Intercept=False)                     10        0.9379         0.8857          0.8500       0.9444         0.8947         0.5234
      LASSO LR                              LASSO LR (L1-saga, C=0.03)                      3        0.9060         0.8429          0.8378       0.8611         0.8493         0.5138

